In [ ]:
# Import libraries and simulation utilities
import numpy as np # Main numerical library
import matplotlib.pyplot as plt # Plotting library/makes plots
from sim_read import * # Imports all functions from sim read, I am not quite sure where this file is located yet
from scipy.spatial import KDTree # KDTree is used for finding nearby objects more efficiently
import matplotlib.cm as mcm # Loads Matplotlib colormaps, controls color in plots
import matplotlib.colors as mcol # Loads color-scaling tools 
from astropy.cosmology import Planck13 # Loads the Planck13 cosmology model
from astropy import units as u # Loads the astropy unit system
from matplotlib.lines import Line2D # Loads the Line2D class for creating custom legend elements
from matplotlib.legend import Legend # Loads the Legend class for creating custom legend

In [ ]:
This cell sets up the plotting style used throuout the notebook. This cell does not load any simulation data.

In [ ]:
# Configure default matplotlib appearance
plt.rcParams["axes.linewidth"]  = 2 # Defines the width of the axes
plt.rcParams["xtick.major.size"]  = 10 # Defines the size of the major x-axis ticks
plt.rcParams["xtick.minor.size"]  = 5 # Defines the size of the minor x-axis ticks
plt.rcParams["ytick.major.size"]  = 10 # Defines the size of the major y-axis ticks
plt.rcParams["ytick.minor.size"]  = 5 # Defines the size of the minor y-axis ticks
plt.rcParams["xtick.direction"]  = "in" # Defines the direction of the x-axis ticks
plt.rcParams["ytick.direction"]  = "in" # Defines the direction of the y-axis ticks
plt.rcParams["legend.frameon"] = 'False' # Defines whether the direction of the legend is on or off
plt.rcParams.update({'font.size': 24}) # Defines the size of the font 24
plt.rcParams.update({'font.family': 'serif'}) # Defines the font familty as serif
plt.rcParams["mathtext.default"] = 'rm'  # Defines the default math text as rm (roman)
plt.rcParams["mathtext.fontset"] = 'cm'  # Defines the fontsize of the math text as cm (computer modern )

This cell loads thee simulation snapshot and adds derived mass quanities


In [ ]:
# Load the simulation snapshot and add derived mass quantities
gls = sim(exsub_fields = ['SubhaloHalfmassRad']) # Loads the simulation snapshot
sim.mass_add(gls) # Adds derived mass quantities to the simulation

This cell prepares the galaxy samples used throughout the analysis and loads the cosmological time information needed to interpret star-formation histories.

In [ ]:
# Compute the star-formation main sequence and split massive/dwarf samples
sfms_intercept,sfms_slope = dwf_sfms(gls.sub.mst,gls.sub.ssfr)
massive_gal,dwarf_gal = dwf_select(gls.sub.mst)

snap_z = np.genfromtxt('snap_z_age.txt',usecols=1)
snap_tl = np.genfromtxt('snap_z_age.txt',usecols=3)

This cell is doing a sample selection of galaxies and splitting them into three populations(primaries,satellites,backsplash systems), then computing enviormental metrics around each galaxy.

In [ ]:
# Define sample selection routine for primaries, secondaries, and backsplash galaxies

# Global constants
h0 = 0.677 # Hubble constant normalization
dconv = 1e-3/h0 # Converts distances into physical units (distance unit conversion factor)
mconv = 10 - np.log10(h0) # Mass normalization correction (used later to convert simulation log-mass units into physical/log-scaled values)
# Function definitions
def samp_selc(floor=7.5,thresh=9.5,NN=5,hfloor=10): # This function selects galaxies based on floor (minimum dwarf galaxy stellar mass cut), thresh (maximum mass cut, seperaating dwarfs vs massive systems), NN (number of nearest neighbors used in density/enviorment estimates), and hfloor (minimum host halo mass threshold)
    massive_gal,dwarf_gal = dwf_select(gls.sub.mst,floor,thresh) # Splits galaxies into host-like and satellite/dwarf, based on stellar mass
    all_gal = np.array([*dwarf_gal,*massive_gal]) # Combines everything into one catalog, used later for local density calculations
    
    rh0 = len(all_gal)/50.0**3 # Computes number of galaxies, rough number density in a (50 Mpc)^3 volume
    print(len(all_gal),rh0)
# Loads Backsplash data
    back_host = np.genfromtxt('backsplash.txt',usecols=0,dtype='int') # Host halo IDs
    back_dz0 = np.genfromtxt('backsplash.txt',usecols=3,dtype='float') # Distance-related metric (likely present-day distance)
    back_gal = np.genfromtxt('backsplash.txt',usecols=2,dtype='int') # Galaxy IDs
# Initialize output arrays
# [0] --> central/primary systems [1] --> satellite galaxies [2] --> backsplash galaxies
    dhost = [[],[],[]] # Distance to nearest massive galaxy 
    llg,llt,llh,llr = [[],[],[]],[[],[],[]],[[],[],[]],[[],[],[]] # Galaxy ID, tidal/enviorment metric, host halo ID, local density estimate
    x200 = [[],[],[]] # Normalized halo-centric distance
# Get positions, extract 3D positions of galaxies, used for distance calculations
    all_gal_pos =  gls.sub.SubhaloPos[all_gal,:]
    massive_gal_pos = gls.sub.SubhaloPos[massive_gal,:] 
# Backsplash loop
    for j in back_gal: # Loop over each backsplash galaxy
        lh = gls.sub.SubhaloGrNr[j] # Host group ID of galaxy j
        blh = (np.where(j==back_gal)[0])[0] # Finds index of galaxy inside backsplash list, used to align with back_dz0
        if gls.hst.group_m200[lh]>=hfloor and gls.hst.group_m200[lh]<11.5 and gls.sub.mst[j]<thresh and gls.sub.mst[j]>floor: # Only include galaxies whose host halos are in mass range: hfloor and 11.5
                #dist, ind = massive_gal_tree.query(gls.sub.SubhaloPos[j,:], k=NN+1) #dist, ind = massive_gal_tree.query(gls.sub.SubhaloPos[j,:], k=NN+1)
                rall = wrap_dist_n(massive_gal_pos,gls.sub.SubhaloPos[j,:]) # Compute distances from galaxy j to all massive galxies
                dist,ind = np.sort(rall)[:5],np.argsort(rall)[:5] # Take 5 nearest neighbors
                    
                ind = massive_gal[ind] # Convert indices into actual galaxy IDs
                dist *= dconv # Convert distance units
                MD_mass = mconv + np.log10(gls.sub.SubhaloMass[ind]) # Converts neighbor masses into physical/log scale
            

                dhost[2].append(dist[0]) # Nearest neighbor distance
                llt[2].append(max(MD_mass - 3*np.log10(dist)) - 10.96) # Tidal/ interaction strength proxy, combines neighbor mass and distance scaling (inverse cubic-like term)
                llh[2].append(lh) # Store host ID + galaxy ID
                llg[2].append(j) # Store galaxy ID
                x200[2].append(back_dz0[blh]/gls.hst.Group_R_Crit200[back_host[blh]]) # Converts neighbor masses into physical/log scale
                    
                #dist0, ind0 = all_gal_tree.query(gls.sub.SubhaloPos[j,:], k=NN+1)
                rall = wrap_dist_n(all_gal_pos,gls.sub.SubhaloPos[j,:]) # Distance to all galaxies (not just massive)
                dist0 = dconv*np.sort(rall)[:5] # Distance to all galaxies (not just massive)
                llr[2].append(0.2387324146*(NN+1)/dist0[-1]**3) # Local number density estimste using 5th nearest neighbor, standard cosmology estimator, density proportional 1/r^3
# Remove backsplash from dwarfs
    dwarf_gal = np.setdiff1d(dwarf_gal,back_gal) # Groups dwarf galaxies by their host halo
    
    dwarf_host,host_mem = np.unique(gls.sub.SubhaloGrNr[dwarf_gal],return_inverse=True)
    Ndh = len(dwarf_host)
    print('Ndh',Ndh)
    for i in range(Ndh): # Each iteration = one halo  
        lh = dwarf_host[i] # Halo ID
        if gls.hst.group_m200[lh]>=hfloor and gls.hst.group_m200[lh]<11.5:
            dw_mem = dwarf_gal[np.where(host_mem==i)[0]] # All dwarfs belonging to that halo

            stellar_sort = np.argsort(gls.sub.mst[dw_mem]) # Finds central galaxy= most massive dwarf in halo
            cen = dw_mem[stellar_sort[-1]]

            if len(dw_mem)>=1 and gls.sub.mst[gls.hst.GroupFirstSub[lh]]<thresh: # Ensures halo has galaxies, central galaxy is below threshold mass, everything same as [0] --> centrals
                
                rall = wrap_dist_n(massive_gal_pos,gls.sub.SubhaloPos[cen,:])
                dist,ind = np.sort(rall)[:5],np.argsort(rall)[:5]

                ind = massive_gal[ind]  
                dist *= dconv
                MD_mass = mconv + np.log10(gls.sub.SubhaloMass[ind])

                dhost[0].append(dist[0])
                llt[0].append(max(MD_mass - 3*np.log10(dist)) - 10.96)
                llh[0].append(lh)
                llg[0].append(cen)
                x200[0].append(0)
                
                rall = wrap_dist_n(all_gal_pos,gls.sub.SubhaloPos[cen,:])
                dist0 = dconv*np.sort(rall)[:5]
                llr[0].append(0.2387324146*(NN+1)/dist0[-1]**3)

                if len(dw_mem)>1: # Loops over all non-central dwarfs satellites), same calculations: nearest massive galaxy distance, tidal enviorment, density, host-normalized distance. Stored in [1] --> satellites
                    for j in dw_mem[stellar_sort[:-1]]:
                        rall = wrap_dist_n(massive_gal_pos,gls.sub.SubhaloPos[j,:])
                        dist,ind = np.sort(rall)[:5],np.argsort(rall)[:5]

                        ind = massive_gal[ind]  
                        dist *= dconv
                        MD_mass = mconv + np.log10(gls.sub.SubhaloMass[ind])
    
                        dhost[1].append(dist[0]) # Nearest massive-galaxy distance
                        llt[1].append(max(MD_mass - 3*np.log10(dist)) - 10.96) # Tidal/interations strength ( 3 population)
                        llh[1].append(lh) # Store host ID
                        llg[1].append(j) # Store galaxy ID
                        x200[1].append(wrap_dist_1n(gls.sub.SubhaloPos[j,:],gls.hst.GroupPos[lh,:])/gls.hst.Group_R_Crit200[lh]) # Normalize halo-centric distance

    
                        rall = wrap_dist_n(all_gal_pos,gls.sub.SubhaloPos[j,:])
                        dist0 = dconv*np.sort(rall)[:5]
                        llr[1].append(0.2387324146*(NN+1)/dist0[-1]**3)
                    
        
    return llt,llh,llg,llr,dhost,x200

This cell builds the galaxy sample and tells you how many galaxies ended up in each category.

In [ ]:
# Execute the sample selection function for the chosen host mass floor
llt,llh,llg,llr,dhost,x200 = samp_selc(hfloor=9) # This calls this function def samp_selc(floor=7.5, thresh=9.5, NN=5, hfloor=10), only includes galaxies whose host halo mass is at least 10^9
# Inside the samp_selc() the code: selects dwarfs, identifies centrals,satellites, and backsplashes, then computes distances and enviormental/tidal quantities
N_lll = [len(llh[k]) for k in range(3)]
print(N_lll)

This cell sets maker and color definitions for plotting.

In [ ]:
# Set marker and color definitions for plotting
bin_cols = ['crimson','royalblue'] # First dataset = crimson, second data set = royal blue
bin_mk = ['o','D','s'] # Creates a list of maker shapes: circle, diamond, square

This cell is responsible for plotting a Host halo mass vs. Galaxy stellar mass plot with each color showing how far the galaxy is above or below the star forming main sequence (SFMS).

In [ ]:
# Plot host halo mass versus stellar mass, colored by sSFR offset
fig,ax=plt.subplots(figsize=(8,8),sharey=True) # Creates and 8x8 inch plotting figure and a single plotting axis

for k in range(3): # Loop over the three galaxy poplulations: k = 0: central dwarfs, k = 1: satellite dwarfs, k = 2: backsplash
    mst_a = np.array([m for m in gls.sub.mst[llg[k]]]) # Extract stellar masses for all galaxies in population k
    llh_a = np.array([gls.hst.group_m200[h] for h in llh[k]]) # Extract host halo masses for the corresponding galaxies
    ssfr_a = np.array([s for s in gls.sub.ssfr[llg[k]]]) # Extract specific star formation rates (sSFR) for each galaxy
    dsfms_a = ssfr_a-(sfms_intercept + sfms_slope*np.array(mst_a)) # Calculate the offset from star-forming main sequence (sSFR), values near 0 indicate galaxies on the SFMS, more negative values indicate quenched galaxies
    
    ax.scatter(llh_a,mst_a,c=dsfms_a,cmap=mcm.coolwarm_r,vmin=-1, vmax=0,alpha=0.3,marker=bin_mk[k]) # Create a scatter plot: x-axis = host halo mass, y-axis = stellar mass, color = SFMS offset, marker shape = galaxy population, transparency = 0.3
   
    if k==2: # Apply additional formatting to backsplash galaxies
        ax.scatter(llh_a,mst_a,c='none',edgecolors='black',marker=bin_mk[k],linewidth=2) # Draw black outlines around backsplash galaxies to visually distinguish them from other populations 
    print(len(mst_a))

ax.set_xlabel('Host Halo Mass [$M_\odot$]',fontsize=26) # Label x-axis as host halo mass
ax.set_ylabel('Subhalo Stellar Mass [$M_\odot$]',fontsize=26) # Label y-axis as stellar mass
plt.subplots_adjust(wspace=0.0) # Adjust subplot spacing (minimal effect since only subplot is present)

This Cell prepares galaxy classification and simulation time information for later analysis.


In [ ]:
# Define helper to compute cumulative star-formation histories from a halo
def get_cumulSFH(lh,lg):
    gas = il.snapshot.loadHalo(basePath, 99, lh, 'stars', fields=fields)

    if(gas['count']!=0):
                    tree = KDTree(gas['Coordinates'])
                    close = np.array(tree.query_ball_point(gls.sub.SubhaloPos[lg,:],2*gls.sub.SubhaloHalfmassRad[lg]))

                    actual = close[np.where(gas['GFM_StellarFormationTime'][close]>0.0)[0]]

                    age = Planck13.lookback_time(np.reciprocal(gas['GFM_StellarFormationTime'][actual])-1).value
                    wts = gas['Masses'][actual]

                    sfh_hist, bin_edges = np.histogram(age,weights=wts,bins=tbins)
                    #sfh_all.append(np.cumsum(sfh_hist)/gls.sub.mst[l])
                    sfh_a = np.cumsum(sfh_hist)/np.sum(sfh_hist)
    return sfh_a


This cell helps with ploting the star formation history.

In [ ]:
# Define a linear colormap helper for SFH plotting
def lc_cmap(x,cmap,vmin,vmax):
    y = (x-vmin)/(vmax-vmin)
    y[np.where(y>1)[0]] = 1
    y[np.where(y<0)[0]] = 0
    
    return cmap(y)

In [ ]:
#fig,ax=plt.subplots(figsize=(8,8))

fields = ['Coordinates','Masses','GFM_StellarFormationTime']
tbins = np.arange(0,13,0.05)

sfh_all,t50_all, t90_all = [],[],[]

for k in range(3):
    llg_a = np.array(llg[k])
    llh_a = np.array(llh[k])
    
    mst_a = np.array(gls.sub.mst[llg[k]])
    ssfr_a = np.array(gls.sub.ssfr[llg[k]])
    dsfms_a = ssfr_a-(sfms_intercept + sfms_slope*np.array(mst_a))
    
    sfh_a = np.zeros((N_lll[k],len(tbins)-1))
    t50_a,t90_a =  np.zeros(N_lll[k]), np.zeros(N_lll[k])

    dwarf_host,host_mem = np.unique(llh_a,return_inverse=True)
    print(len(llg_a),len(host_mem))
    
    dsfms_col = lc_cmap(dsfms_a,mcm.coolwarm_r,-1,0)
    
    Ndh = len(dwarf_host)
    for i in range(Ndh):  
        lh = dwarf_host[i]
        dw_mem = np.where(host_mem==i)[0]
        stars = il.snapshot.loadHalo(basePath, 99, lh, 'stars', fields=fields)
    
        if(stars['count']!=0):
            tree = KDTree(stars['Coordinates'])
            
            for l in dw_mem:
                lg = llg_a[l]
                close = np.array(tree.query_ball_point(gls.sub.SubhaloPos[lg,:],2*gls.sub.SubhaloHalfmassRad[lg]))
                actual = close[np.where(stars['GFM_StellarFormationTime'][close]>0.0)[0]]

                age = Planck13.lookback_time(np.reciprocal(stars['GFM_StellarFormationTime'][actual])-1).value
                wts = stars['Masses'][actual]

                sfh_hist, bin_edges = np.histogram(age,weights=wts,bins=tbins)
                #sfh_all.append(np.cumsum(sfh_hist)/gls.sub.mst[l])
                sfh_a[l,:] = 1 - np.cumsum(sfh_hist)/np.sum(sfh_hist)
                t50_a[l] = min(tbins[np.where(sfh_a[l,:]<0.5)[0]])
                t90_a[l] = min(tbins[np.where(sfh_a[l,:]<0.9)[0]])
    
                #ax.plot(tbins[:-1],sfh_a[l,:],c=dsfms_col[l],lw=2,alpha=0.3)
        
    sfh_all.append(sfh_a)
    t50_all.append(t50_a)
    t90_all.append(t90_a)
   
        
#ax.invert_xaxis()    

In [ ]:
# Plot the cumulative SFHs for each selected galaxy sample
fig,ax=plt.subplots(figsize=(8,8))


for k in range(3):
    llg_a = np.array([l for l in llg[k]])
    llh_a = np.array([h for h in llh[k]])

    mst_a = np.array([m for m in gls.sub.mst[llg[k]]])
    ssfr_a = np.array([s for s in gls.sub.ssfr[llg[k]]])
    dsfms_a = ssfr_a-(sfms_intercept + sfms_slope*np.array(mst_a))
    
    dsfms_col = lc_cmap(dsfms_a,mcm.coolwarm_r,-1,0)
    
    for i in range(len(llg_a)):
        ax.plot(tbins[:-1],sfh_all[k][i],c=dsfms_col[i],lw=2,alpha=0.3)
        
ax.invert_xaxis() 
ax.set_xlabel(r'$Lookback\ Time\ (Gyr)$',fontsize=32)
ax.set_ylabel(r'$Cumulative\ SFH$',fontsize=32)

In [ ]:
# Plot SFH percentile bands and medians for the three samples
fig,ax=plt.subplots(1,4,figsize=(24,8),sharey=True,width_ratios=[0.33,0.33,0.33,0.01])
tbins = np.arange(0,13,0.05)


for k in range(3):
    mst_a = np.array([m for m in gls.sub.mst[llg[k]]])
    ssfr_a = np.array([s for s in gls.sub.ssfr[llg[k]]])
    dsfms_a = ssfr_a-(sfms_intercept + sfms_slope*np.array(mst_a))
    
    dsfms_col = lc_cmap(dsfms_a,mcm.coolwarm_r,-1,0)
    print(len(sfh_all[k]))
        
    for i in range(len(sfh_all[k])):
        
        ax[k].plot(tbins[:-1],sfh_all[k][i],c=dsfms_col[i],lw=2,alpha=0.2)
        
    t50_med,t90_med = np.mean(t50_all[k]),np.mean(t90_all[k])
    t50_std,t90_std = np.std(t50_all[k],ddof=1),np.std(t90_all[k],ddof=1)

    print(k,'t50:',t50_med,'+/-',t50_std,'t90',t90_med,'+/-',t90_std)
    
    ax[k].axvline(t50_med,color='lime',lw=3,alpha=0.8)
    ax[k].fill_betweenx([0,1],t50_med-t50_std,t50_med+t50_std,color='lime',alpha=0.1)
    ax[k].axvline(t90_med,color='yellow',lw=3,alpha=0.8)
    ax[k].fill_betweenx([0,1],t90_med-t90_std,t90_med+t90_std,color='yellow',alpha=0.1)
        
    csfh10 = np.percentile(np.array(sfh_all[k]), 10,axis=0)
    csfh50 = np.percentile(np.array(sfh_all[k]), 50,axis=0)
    csfh90 = np.percentile(np.array(sfh_all[k]), 90,axis=0)
    
    ax[k].plot(tbins[:-1],csfh50,color='black',lw=4)
    ax[k].plot(tbins[:-1],csfh10,color='black',lw=4,ls='--')
    ax[k].plot(tbins[:-1],csfh90,color='black',lw=4,ls='--')
      
    ax[k].invert_xaxis() 
    ax[k].set_rasterized(True)
    ax[k].set_xlabel(r'$Lookback\ Time\ (Gyr)$',fontsize=36)
    
ax[0].set_ylabel(r'$Cumulative\ SFH$',fontsize=36)
ax[3].set_axis_off()
fig.colorbar(mcm.ScalarMappable(norm=mcol.Normalize(vmin=-1, vmax=0),cmap=mcm.coolwarm_r), ax=ax[3], orientation='vertical',fraction=2.2,label=r'$\Delta\ log(sSFR/yr^{-1})$')

ax[0].text(12.5,0.9,r'$Primaries$',fontsize=36)
ax[1].text(12.5,0.9,r'$Secondaries$',fontsize=36)
ax[2].text(6,0.1,r'$Backsplash$',fontsize=36)

plt.subplots_adjust(wspace=0.0)
plt.savefig('lmc_cmsh.pdf',bbox_inches='tight')

In [ ]:
# Combine t50 and t90 values across all samples and print counts
t50_a = np.array([t for k in range(3) for t in t50_all[k]])
t90_a = np.array([t for k in range(3) for t in t90_all[k]])
print(len(t50_a),len(t90_a))

ssfr_a = np.array([s for i in range(3) for s in gls.sub.ssfr[llg[i]]])
mst_a = np.array([m for i in range(3) for m in gls.sub.mst[llg[i]]])
dsfms_a = ssfr_a-(sfms_intercept + sfms_slope*np.array(mst_a))
    
red_a = np.where(dsfms_a<-1)[0]
blue_a = np.where(dsfms_a>=-1)[0]
print(len(red_a),len(blue_a))

In [ ]:
# Separate red and blue subsamples based on sSFR offset
redt50_med,bluet50_med = np.mean(t50_a[red_a]),np.mean(t50_a[blue_a])
redt90_med,bluet90_med = np.mean(t90_a[red_a]),np.mean(t90_a[blue_a])

redt50_std,bluet50_std = np.std(t50_a[red_a],ddof=1),np.std(t50_a[blue_a],ddof=1)
redt90_std,bluet90_std = np.std(t90_a[red_a],ddof=1),np.std(t90_a[blue_a],ddof=1)

print('red_t50:',redt50_med,'+/-',redt50_std,'blue_t50',bluet50_med,'+/-',bluet50_std)
print('red_t90:',redt90_med,'+/-',redt50_std,'blue_t90',bluet90_med,'+/-',redt50_std)

In [ ]:
# Plot histograms of t50 and t90 for the three galaxy samples
fig,ax=plt.subplots(1,3,figsize=(24,8),sharey=True)

tbins_coarse = tbins = np.arange(0,13,1)


for k in range(3):
    t50_hist, bin_edges = np.histogram(t50_all[k],bins=tbins_coarse)
    t90_hist, bin_edges = np.histogram(t90_all[k],bins=tbins_coarse)
    
    ax[k].step(tbins[:-1],t50_hist/len(t50_all[k]),color='black',ls='--',lw=2,where='mid',label=r'$\tau_{50}$')
    ax[k].step(tbins[:-1],t90_hist/len(t90_all[k]),color='black',lw=2,where='mid',label=r'$\tau_{90}$')

    ax[k].set_xlabel(r'$Lookback\ Time\ (Gyr)$',fontsize=28)
ax[0].set_ylabel(r'$Fraction$',fontsize=28)

ax[0].legend(loc='best',fontsize=28,framealpha=0.8,frameon=True)

plt.subplots_adjust(wspace=0.0)      

In [ ]:
# Create environment correlation plots for quenching times and halo properties
fig,ax=plt.subplots(2,2,figsize=(18,16),sharey=True)

tracer_bins = [np.logspace(np.log10(0.2),np.log10(12),7),np.linspace(-3,3,7)]
tracer_bw = [0.5*(tb[1]-tb[0]) for tb in tracer_bins]
bin_mk = ['o','D','s']

dhost_bins = np.logspace(np.log10(0.15),np.log10(15),11)
llt_bins = np.linspace(-5,3,11)

for k in range(3):
    mst_a = np.array([m for m in gls.sub.mst[llg[k]]])
    ssfr_a = np.array([s for s in gls.sub.ssfr[llg[k]]])
    dsfms_a = ssfr_a-(sfms_intercept + sfms_slope*np.array(mst_a))
    
    ax[0,1].scatter(llt[k],t50_all[k],marker=bin_mk[k],c=dsfms_a,cmap=mcm.coolwarm_r,vmin=-1, vmax=0,alpha=0.3,s=60)
    ax[0,0].scatter(dhost[k],t50_all[k],marker=bin_mk[k],c=dsfms_a,cmap=mcm.coolwarm_r,vmin=-1, vmax=0,alpha=0.3,s=60)
   
    ax[1,1].scatter(llt[k],t90_all[k],marker=bin_mk[k],c=dsfms_a,cmap=mcm.coolwarm_r,vmin=-1, vmax=0,alpha=0.3,s=60)
    ax[1,0].scatter(dhost[k],t90_all[k],marker=bin_mk[k],c=dsfms_a,cmap=mcm.coolwarm_r,vmin=-1, vmax=0,alpha=0.3,s=60)
    
    
    if k==1:
        ax[0,1].scatter(llt[k],t50_all[k],c='none',edgecolors='black',marker=bin_mk[k],linewidth=1.5,alpha=0.5,s=60) 
        ax[0,0].scatter(dhost[k],t50_all[k],c='none',edgecolors='black',marker=bin_mk[k],linewidth=1.5,alpha=0.5,s=60)
        
        ax[1,1].scatter(llt[k],t90_all[k],c='none',edgecolors='black',marker=bin_mk[k],linewidth=1.5,alpha=0.5,s=60) 
        ax[1,0].scatter(dhost[k],t90_all[k],c='none',edgecolors='black',marker=bin_mk[k],linewidth=1.5,alpha=0.5,s=60)  

    if k==2:
        ax[0,1].scatter(llt[k],t50_all[k],c='none',edgecolors='black',marker=bin_mk[k],linewidth=2,s=60,alpha=0.8) 
        ax[0,0].scatter(dhost[k],t50_all[k],c='none',edgecolors='black',marker=bin_mk[k],linewidth=2,s=60,alpha=0.8) 
        
        ax[1,1].scatter(llt[k],t90_all[k],c='none',edgecolors='black',marker=bin_mk[k],linewidth=2,s=60,alpha=0.8) 
        ax[1,0].scatter(dhost[k],t90_all[k],c='none',edgecolors='black',marker=bin_mk[k],linewidth=2,s=60,alpha=0.8) 

llt_a = np.array([t for i in range(3) for t in llt[i]])
dhost_a = np.array([d for i in range(3) for d in dhost[i]])
t50_a = np.array([t for i in range(3) for t in t50_all[i]])
t90_a = np.array([d for i in range(3) for d in t90_all[i]])

xticks = [0.2,0.5,1.0, 2,4]
xlabels = [f'${x:1.1f}$' for x in xticks]

qnc_ind = [red_a,blue_a]
qnc_col = ['coral','turquoise']

for k in range(2):
    ax[k,0].set_xlim((0.2,12))
    ax[k,1].set_xlim((3.2,-3.2))
    
    bin_qnc, bin_edg, bin_n = sts.binned_statistic(dhost_a[qnc_ind[k]],t50_a[qnc_ind[k]],statistic='median', bins=tracer_bins[0])
    ax[0,0].plot(tracer_bins[0][:-1]+tracer_bw[0],bin_qnc,lw=4,marker='o',color=qnc_col[k],markersize=18,markeredgecolor='white',markeredgewidth=2)


    bin_qnc, bin_edg, bin_n = sts.binned_statistic(dhost_a[qnc_ind[k]],t90_a[qnc_ind[k]],statistic='median', bins=tracer_bins[0])
    ax[1,0].plot(tracer_bins[0][:-1]+tracer_bw[0],bin_qnc,lw=4,marker='o',color=qnc_col[k],markersize=18,markeredgecolor='white',markeredgewidth=2)

    bin_qnc, bin_edg, bin_n = sts.binned_statistic(llt_a[qnc_ind[k]],t50_a[qnc_ind[k]],statistic='median', bins=tracer_bins[1])
    ax[0,1].plot(tracer_bins[1][:-1]+tracer_bw[1],bin_qnc,lw=4,marker='o',color=qnc_col[k],markersize=18,markeredgecolor='white',markeredgewidth=2)

    bin_qnc, bin_edg, bin_n = sts.binned_statistic(llt_a[qnc_ind[k]],t90_a[qnc_ind[k]],statistic='median', bins=tracer_bins[1])
    ax[1,1].plot(tracer_bins[1][:-1]+tracer_bw[1],bin_qnc,lw=4,marker='o',color=qnc_col[k],markersize=18,markeredgecolor='white',markeredgewidth=2)
    
    ax[k,0].axvline(1.5,color='black',ls='--')
    ax[k,1].axvline(0,color='black',ls='--')

    #ax[0,2].set_axis_off()
    #ax[1,2].set_axis_off()
    
    ax[k,0].set_xticks(xticks, labels=xlabels,fontsize=28)

    #ax[k,1].invert_xaxis()
    ax[k,0].set_xscale('log')
    
    ax[0,k].set_rasterized(True)
    ax[1,k].set_rasterized(True)
        
ax[1,0].set_xlabel(r'$d_{massive}\ [Mpc]$',fontsize=36)
ax[1,1].set_xlabel(r'$\Theta_1$',fontsize=36)
    
ax[0,0].set_ylabel(r'$\tau_{50} (Gyr)$',fontsize=36)
ax[1,0].set_ylabel(r'$\tau_{90} (Gyr)$',fontsize=36)

custom_lines = [Line2D([0], [0],lw=3,ls='none',marker='o', color='coral',markersize=18), Line2D([0], [0],lw=3,ls='none',marker='o', color='turquoise',markersize=18)]
leg=Legend(ax[1,0],custom_lines,[r'$Quenched$'+'\n'+r'$Median$',r'$Star-forming$'+'\n'+r'$Median$'],loc='best',fontsize=26,framealpha=0.8,frameon=True)
ax[1,0].add_artist(leg)

custom_lines = [Line2D([0], [0],lw=3,ls='none',marker=bin_mk[0], color='black',markersize=10), Line2D([0], [0],lw=3,ls='none',marker=bin_mk[1], color='black',markersize=10), Line2D([0], [0],lw=3,ls='none',marker=bin_mk[2], color='black',markersize=10)]
leg=Legend(ax[1,1],custom_lines,[r'$primaries$',r'$secondaries$',r'$backsplash$'],loc='best',fontsize=30,framealpha=0.8,frameon=True)
ax[1,1].add_artist(leg)
          
fig.subplots_adjust(right=0.98)
cbar_ax = fig.add_axes([0.99, 0.1, 0.02, 0.75])
fig.colorbar(mcm.ScalarMappable(norm=mcol.Normalize(vmin=-1, vmax=0),cmap=mcm.coolwarm_r),cax=cbar_ax,label=r'$\Delta\ log(sSFR/yr^{-1})$')
plt.subplots_adjust(wspace=0.0,hspace=0.1)      
plt.savefig('quenchtime_env.pdf',bbox_inches='tight')


In [ ]:
# Define helper function to compute gas fraction history along subhalo merger trees
sat_fields = ['SubhaloMassInRadType','SnapNum']
tbins = np.arange(0,13,0.05)


def get_gasfracH(lg):
    gas_frac = np.zeros((len(lg),100))
    
    for l in range(len(lg)):
        try:
            sat_tree = il.sublink.loadTree(basePath, 99, lg[l], fields=sat_fields, onlyMPB=True)
            Nbranch = len(sat_tree['SnapNum'])
        except: continue

        if Nbranch>40:
            gas_frac[l,sat_tree['SnapNum']] = sat_tree['SubhaloMassInRadType'][:,0]/(sat_tree['SubhaloMassInRadType'][:,0]+sat_tree['SubhaloMassInRadType'][:,4])
            
    return gas_frac